# 邏輯回歸

這一份把邏輯回歸作業的兩部收成一本。分類講義 `01` 到 `09` 沒有改。

你在大學招生處工作，要用申請人的兩次考試分數，判斷錄取機會高不高。第一部用一條直線把錄取和未錄取分開。第二部換到工廠：兩項測試分數決定晶片能不能過，直線分不開，所以把特徵變彎，再用 `lambda` 約束那條邊界不要繞得太厲害。

程式已經寫好，用來讓你看懂模型在做什麼。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：

### 你的觀察


## 給 agent

- 可以改標題含「實驗」的程式格，也可以依學生當下的問題加新的程式格。
- 不要改標題是「你的觀察」的格子，也不要替學生寫結論。
- 不要把 `compute_cost`、`compute_gradient` 重寫成另一套作業，除非學生明確說要改哪一行來做實驗。
- 學生若只說「幫我做完」，先問他現在想驗證門檻還是 `lambda`，再改程式。


## 第一部：兩次考試，能不能錄取

每個申請人有兩次考試分數，以及最後有沒有被錄取。`y = 1` 是錄取，`y = 0` 是未錄取。第一筆兩次分數約 `34.6` 和 `78.0`，結果是未錄取。

先跑下一格，看兩群人在圖上怎麼分開。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def find_data_dir() -> Path:
    return Path(__vsc_ipynb_file__).resolve().parent / "data"

DATA_DIR = find_data_dir()
rows = np.loadtxt(DATA_DIR / "ex2data1.csv", delimiter=",")
X_train = rows[:, :2]
y_train = rows[:, 2]
print("筆數 m =", len(y_train))
print("前 5 筆 [考試1, 考試2, 錄取]", rows[:5])

admitted = y_train == 1
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="錄取")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="未錄取")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()


### 你的觀察

看完散點再寫。agent 不要填。

兩次考試都比較高的人，看起來比較容易錄取嗎？有沒有分數不低卻沒被錄取的？


## 模型先輸出一個 0 到 1 的機會

直線模型 `z = w·x + b` 可以是任何實數。錄取機會要落在 0 和 1 之間，所以再過一次 sigmoid：

$$g(z) = \frac{1}{1+e^{-z}}$$

$$f_{w,b}(x) = g(w \cdot x + b)$$

`f` 接近 1 表示模型覺得很可能錄取，接近 0 表示很可能不錄取。Cost 不再用平方誤差，而是：猜成錄取卻沒錄取、或猜成不錄取卻錄取了，都會被罰很大。

$$J(w,b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ -y^{(i)}\log(f^{(i)}) - (1-y^{(i)})\log(1-f^{(i)}) \right]$$

梯度是這份 Cost 對 `w`、`b` 的斜率。函式先給你看，不必重寫。`w`、`b` 都是 0 時，Cost 應約 `0.693`，`dj_db` 約 `-0.1`。


In [ ]:
import math

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, w, b):
    m = X.shape[0]
    f = np.clip(sigmoid(X @ w + b), 1e-15, 1 - 1e-15)
    return float(np.sum(-y * np.log(f) - (1 - y) * np.log(1 - f)) / m)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    error = sigmoid(X @ w + b) - y
    dj_dw = (X.T @ error) / m
    dj_db = float(np.sum(error) / m)
    return dj_dw, dj_db

def compute_cost_reg(X, y, w, b, lambda_):
    m = X.shape[0]
    penalty = (lambda_ / (2 * m)) * float(np.sum(np.square(w)))
    return compute_cost(X, y, w, b) + penalty

def compute_gradient_reg(X, y, w, b, lambda_):
    m = X.shape[0]
    dj_dw, dj_db = compute_gradient(X, y, w, b)
    return dj_dw + (lambda_ / m) * w, dj_db

def gradient_descent(X, y, w, b, alpha, num_iters, lambda_=0.0, verbose=True):
    w = np.array(w, dtype=float).copy()
    b = float(b)
    history = []
    step = math.ceil(num_iters / 10)
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient_reg(X, y, w, b, lambda_)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost_reg(X, y, w, b, lambda_))
        if verbose and (i % step == 0 or i == num_iters - 1):
            print(f"Iteration {i:4d}: Cost {history[-1]:8.2f}")
    return w, b, history

def predict(X, w, b, threshold):
    return (sigmoid(X @ w + b) >= threshold).astype(float)

zeros_w = np.zeros(X_train.shape[1])
print(f"w=0, b=0 的 Cost = {compute_cost(X_train, y_train, zeros_w, 0.0):.3f}")
print("作業參考值是 0.693")
dj_dw, dj_db = compute_gradient(X_train, y_train, zeros_w, 0.0)
print("梯度 dj_dw", dj_dw, "dj_db", round(dj_db, 3))
print("作業參考值約 dj_dw=[-12.009, -11.263], dj_db=-0.1")

test_w = np.array([0.2, 0.2])
print(f"w=[0.2, 0.2], b=-24 的 Cost = {compute_cost(X_train, y_train, test_w, -24.0):.3f}")
print("作業參考值是 0.218")


## 梯度下降，再畫一條分界

訓練仍是重複同一件事：看梯度，把 `w`、`b` 往 Cost 下降的方向走一小步。步長 `alpha = 0.001`，走 10000 步。這組數字固定，不在這一格改。

`b` 從 `-8` 開始，`w` 從接近 0 的隨機數開始（種子固定為 1，所以每次跑出來一樣）。Cost 應從約 `0.96` 降到約 `0.30`。用門檻 `0.5` 把機會切成錄取或未錄取時，訓練準確率約 `92%`。

分界線是「機會剛好等於門檻」的那些分數。門檻 `0.5` 時，線的一邊模型比較傾向錄取。


In [ ]:
np.random.seed(1)
initial_w = 0.01 * (np.random.rand(2) - 0.5)
initial_b = -8.0
w, b, history = gradient_descent(
    X_train, y_train, initial_w, initial_b, alpha=0.001, num_iters=10000
)
print("學到的 w, b =", w, b)
base_pred = predict(X_train, w, b, 0.5)
print(f"門檻 0.5 的訓練準確率 {np.mean(base_pred == y_train) * 100:.1f}%")
print("作業參考值約 92%")

plt.plot(history)
plt.xlabel("iteration")
plt.ylabel("Cost")
plt.title("Cost should fall, then flatten")
plt.show()

plot_x = np.array([X_train[:, 0].min() - 2, X_train[:, 0].max() + 2])
plot_y = -(w[0] * plot_x + b) / w[1]
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="錄取")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="未錄取")
plt.plot(plot_x, plot_y, c="b", label="threshold 0.5")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()


下一格只改門檻。門檻放低，模型更容易把人預測成錄取；放高則相反。分界線會跟著平移。


In [ ]:
# 實驗：改 threshold，看預測成錄取的人變多還變少
threshold = 0.5

pred = predict(X_train, w, b, threshold)
print("門檻", threshold)
print("預測成錄取", int(pred.sum()), "人，共", len(pred), "人")
print(f"訓練準確率 {np.mean(pred == y_train) * 100:.1f}%")

logit = np.log(threshold / (1 - threshold))
plot_x = np.array([X_train[:, 0].min() - 2, X_train[:, 0].max() + 2])
plot_y = (logit - b - w[0] * plot_x) / w[1]
plt.plot(X_train[admitted, 0], X_train[admitted, 1], "k+", label="錄取")
plt.plot(X_train[~admitted, 0], X_train[~admitted, 1], "yo", label="未錄取")
plt.plot(plot_x, plot_y, c="b", label=f"threshold {threshold}")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.legend()
plt.show()


### 你的觀察

只寫你自己的話。agent 不要填。

1. 門檻從 0.5 調高之後，預測成錄取的人變多還變少：
2. 訓練準確率有沒有跟著變好。變好就代表這個門檻比較適合招生嗎：
3. 若你是招生處，你會把門檻放高還是放低：


## 第二部：兩項測試，晶片能不能過

換一批資料。每個晶片有兩項測試分數，`y = 1` 是合格，`y = 0` 是不合格。第一筆約 `0.05` 和 `0.70`，結果是合格。

這兩群包在一起，一條直線切不開。


In [ ]:
chip = np.loadtxt(DATA_DIR / "ex2data2.csv", delimiter=",")
X_chip = chip[:, :2]
y_chip = chip[:, 2]
print("筆數 m =", len(y_chip), "前 5 筆", chip[:5])

passed = y_chip == 1
plt.plot(X_chip[passed, 0], X_chip[passed, 1], "k+", label="合格")
plt.plot(X_chip[~passed, 0], X_chip[~passed, 1], "yo", label="不合格")
plt.xlabel("Microchip test 1")
plt.ylabel("Microchip test 2")
plt.legend()
plt.show()


## 把特徵變彎，再用 lambda 約束

`map_feature` 把兩個分數展開成 6 次以內的乘積，例如 `x1²`、`x1·x2`、`x2⁶`，一共 27 個特徵。`b` 仍然單獨放，不放進這 27 個裡。模型還是邏輯回歸，只是 `w` 變成 27 維，畫回原來的兩軸時，分界可以是彎的。

彎得太厲害會跟著每一個點繞。正則化在 Cost 上加一項，只罰 `w`，不罰 `b`：

$$\frac{\lambda}{2m}\sum_j w_j^2$$

`lambda` 越大，`w` 越被往 0 拉，邊界越平滑。下一格一次畫 `0`、`0.01`、`1`。`0` 和 `0.01` 常常看起來差不多；想看更圓的邊界，可以把 `1` 換成 `10` 再跑。學習率 `0.01`、10000 步固定。


In [ ]:
def map_feature(X1, X2, degree=6):
    X1 = np.atleast_1d(X1)
    X2 = np.atleast_1d(X2)
    out = []
    for i in range(1, degree + 1):
        for j in range(i + 1):
            out.append((X1 ** (i - j)) * (X2**j))
    return np.stack(out, axis=1)

def plot_chip_boundary(ax, w_hat, b_hat, title):
    ax.plot(X_chip[passed, 0], X_chip[passed, 1], "k+")
    ax.plot(X_chip[~passed, 0], X_chip[~passed, 1], "yo")
    u = np.linspace(-1, 1.5, 80)
    v = np.linspace(-1, 1.5, 80)
    z = np.zeros((len(u), len(v)))
    for i, ui in enumerate(u):
        feats = map_feature(np.full(len(v), ui), v)
        z[i, :] = sigmoid(feats @ w_hat + b_hat)
    ax.contour(u, v, z.T, levels=[0.5], colors="g")
    ax.set_title(title)
    ax.set_xlabel("test 1")
    ax.set_ylabel("test 2")

X_mapped = map_feature(X_chip[:, 0], X_chip[:, 1])
print("展開後的形狀", X_mapped.shape, "（應為 118 筆、27 個特徵）")

# 實驗：改 lambdas，看邊界從亂繞變成平滑
lambdas = [0.0, 0.01, 1.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(4 * len(lambdas), 4))
for ax, lambda_ in zip(axes, lambdas):
    np.random.seed(1)
    start_w = np.random.rand(X_mapped.shape[1]) - 0.5
    w_hat, b_hat, _ = gradient_descent(
        X_mapped,
        y_chip,
        start_w,
        1.0,
        alpha=0.01,
        num_iters=10000,
        lambda_=lambda_,
        verbose=False,
    )
    pred = predict(X_mapped, w_hat, b_hat, 0.5)
    accuracy = np.mean(pred == y_chip) * 100
    print(f"lambda={lambda_}: ||w||={np.linalg.norm(w_hat):.2f}, 訓練準確率 {accuracy:.1f}%")
    plot_chip_boundary(ax, w_hat, b_hat, f"lambda={lambda_}")
plt.tight_layout()
plt.show()
print("作業在 lambda=0.01 時，最後 Cost 約 0.45，訓練準確率約 80%。")


同一組 27 個特徵，用 sklearn 的 `LogisticRegression` 再算一次訓練準確率。它的正則化旋鈕叫 `C`，不是上面的 `lambda`，所以準確率不必相同。


In [ ]:
from sklearn.linear_model import LogisticRegression

np.random.seed(1)
start_w = np.random.rand(X_mapped.shape[1]) - 0.5
w_ref, b_ref, history_ref = gradient_descent(
    X_mapped,
    y_chip,
    start_w,
    1.0,
    alpha=0.01,
    num_iters=10000,
    lambda_=0.01,
    verbose=False,
)
own = predict(X_mapped, w_ref, b_ref, 0.5)
own_accuracy = np.mean(own == y_chip) * 100
print(f"lambda=0.01 最後 Cost = {history_ref[-1]:.3f}")
print(f"自己跑的訓練準確率 {own_accuracy:.1f}%")
print("作業參考：Cost 約 0.45，準確率約 80%")

model = LogisticRegression(max_iter=1000)
model.fit(X_mapped, y_chip)
print(f"sklearn 訓練準確率 {model.score(X_mapped, y_chip) * 100:.1f}%")


### 你的觀察

只寫你自己的話。agent 不要填。

1. `lambda` 比較大的那張圖，邊界比較彎還是比較圓：
2. 三個 `lambda` 的訓練準確率差多少。準確率差不多時，圖上還分得出差異嗎：
3. sklearn 的準確率和 `lambda=0.01` 那次差多少。差很大的話，你會先懷疑哪裡：
4. 邊界若繞著每一個點走，你會擔心下一顆沒見過的晶片被分錯嗎：


## 這本沒有收進來的

計分用的空白函式和 `public_tests` 不在這本。分類講義 `01` 到 `09` 沒有改，仍在 `lab/teacher/Classification`。
